In [ ]:
{
 "cells": [
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": [
    "# Convolutional Neural Networks and Regularization\n",
    "\n",
    "This notebook explores Basic DNNs, Regularization, and Convolutional Neural Networks using PyTorch on the MNIST dataset.\n"
   ],
   "id": "1b1f64b98641c442"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": [
    "import time\n",
    "import matplotlib.pyplot as plt\n",
    "import numpy as np\n",
    "import torch\n",
    "import torch.nn as nn\n",
    "import torch.optim as optim\n",
    "from torch.utils.data import DataLoader, Subset, random_split\n",
    "from torchvision import datasets, transforms\n",
    "from torchinfo import summary\n",
    "from tqdm import tqdm\n",
    "\n",
    "# Set seed for reproducibility\n",
    "torch.manual_seed(42)\n",
    "np.random.seed(42)\n",
    "\n",
    "# Device configuration\n",
    "device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n",
    "print(f\"Using device: {device}\")\n"
   ],
   "id": "c3fe53112b0bc289"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": [
    "## Data Preparation\n",
    "\n",
    "We load the MNIST dataset and split the training set into training and validation sets.\n",
    "Original TF split: 49800 train, 10200 val, 10000 test.\n",
    "MNIST total train is 60000. 60000 * 0.17 = 10200. So 60000 - 10200 = 49800. Matches.\n"
   ],
   "id": "b8c8740b08dcaee9"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": [
    "transform = transforms.Compose([\n",
    "    transforms.ToTensor(),\n",
    "])\n",
    "\n",
    "# Load datasets\n",
    "full_train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)\n",
    "test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)\n",
    "\n",
    "# Split train into train and val\n",
    "train_size = 49800\n",
    "val_size = 10200\n",
    "train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])\n",
    "\n",
    "print(f\"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}\")\n",
    "\n",
    "# Data loaders\n",
    "batch_size = 128\n",
    "train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)\n",
    "val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)\n",
    "test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)\n"
   ],
   "id": "ecb8139f43d02d22"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": "## Training Helper Function\n",
   "id": "b5e9d94e74d5266"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": [
    "def train_model(model, train_loader, val_loader, epochs, lr=0.001, weight_decay=0.0):\n",
    "    model.to(device)\n",
    "    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)\n",
    "    criterion = nn.CrossEntropyLoss()\n",
    "    \n",
    "    history = {\n",
    "        'sparse_categorical_accuracy': [],\n",
    "        'val_sparse_categorical_accuracy': [],\n",
    "        'loss': [],\n",
    "        'val_loss': []\n",
    "    }\n",
    "    \n",
    "    for epoch in range(epochs):\n",
    "        model.train()\n",
    "        running_loss = 0.0\n",
    "        correct = 0\n",
    "        total = 0\n",
    "        \n",
    "        for images, labels in tqdm(train_loader, desc=f\"Epoch {epoch+1}/{epochs}\", leave=False):\n",
    "            images, labels = images.to(device), labels.to(device)\n",
    "            \n",
    "            optimizer.zero_grad()\n",
    "            outputs = model(images)\n",
    "            loss = criterion(outputs, labels)\n",
    "            loss.backward()\n",
    "            optimizer.step()\n",
    "            \n",
    "            running_loss += loss.item() * images.size(0)\n",
    "            _, predicted = torch.max(outputs.data, 1)\n",
    "            total += labels.size(0)\n",
    "            correct += (predicted == labels).sum().item()\n",
    "            \n",
    "        train_loss = running_loss / total\n",
    "        train_acc = correct / total\n",
    "        \n",
    "        # Validation\n",
    "        model.eval()\n",
    "        val_running_loss = 0.0\n",
    "        val_correct = 0\n",
    "        val_total = 0\n",
    "        with torch.no_grad():\n",
    "            for images, labels in val_loader:\n",
    "                images, labels = images.to(device), labels.to(device)\n",
    "                outputs = model(images)\n",
    "                loss = criterion(outputs, labels)\n",
    "                \n",
    "                val_running_loss += loss.item() * images.size(0)\n",
    "                _, predicted = torch.max(outputs.data, 1)\n",
    "                val_total += labels.size(0)\n",
    "                val_correct += (predicted == labels).sum().item()\n",
    "                \n",
    "        val_loss = val_running_loss / val_total\n",
    "        val_acc = val_correct / val_total\n",
    "        \n",
    "        history['loss'].append(train_loss)\n",
    "        history['sparse_categorical_accuracy'].append(train_acc)\n",
    "        history['val_loss'].append(val_loss)\n",
    "        history['val_sparse_categorical_accuracy'].append(val_acc)\n",
    "        \n",
    "        print(f\"Epoch {epoch+1}: Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}\")\n",
    "        \n",
    "    return history\n",
    "\n",
    "def evaluate_model(model, test_loader):\n",
    "    model.eval()\n",
    "    model.to(device)\n",
    "    correct = 0\n",
    "    total = 0\n",
    "    criterion = nn.CrossEntropyLoss()\n",
    "    running_loss = 0.0\n",
    "    with torch.no_grad():\n",
    "        for images, labels in test_loader:\n",
    "            images, labels = images.to(device), labels.to(device)\n",
    "            outputs = model(images)\n",
    "            loss = criterion(outputs, labels)\n",
    "            running_loss += loss.item() * images.size(0)\n",
    "            _, predicted = torch.max(outputs.data, 1)\n",
    "            total += labels.size(0)\n",
    "            correct += (predicted == labels).sum().item()\n",
    "    \n",
    "    test_loss = running_loss / total\n",
    "    test_acc = correct / total\n",
    "    print(f\"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}\")\n",
    "    return test_loss, test_acc\n"
   ],
   "id": "724de5f67d6a5a38"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": "## 1. Basic DNN\n",
   "id": "7d186ec6b63a1644"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": [
    "class BasicDNN(nn.Module):\n",
    "    def __init__(self):\n",
    "        super(BasicDNN, self).__init__()\n",
    "        self.flatten = nn.Flatten()\n",
    "        self.fc1 = nn.Linear(28 * 28, 384)\n",
    "        self.relu1 = nn.ReLU()\n",
    "        self.fc2 = nn.Linear(384, 128)\n",
    "        self.relu2 = nn.ReLU()\n",
    "        self.fc3 = nn.Linear(128, 10)\n",
    "        \n",
    "    def forward(self, x):\n",
    "        x = self.flatten(x)\n",
    "        x = self.fc1(x)\n",
    "        x = self.relu1(x)\n",
    "        x = self.fc2(x)\n",
    "        x = self.relu2(x)\n",
    "        x = self.fc3(x)\n",
    "        return x\n",
    "\n",
    "model_basic = BasicDNN()\n",
    "summary(model_basic, input_size=(batch_size, 1, 28, 28))\n"
   ],
   "id": "a8f477f89d58f218"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": [
    "all_histories = []\n",
    "\n",
    "print(\"Training Basic DNN...\")\n",
    "history_basic = train_model(model_basic, train_loader, val_loader, epochs=10)\n",
    "all_histories.append(('Basic DNN', history_basic))\n"
   ],
   "id": "4fd51a2c5a148dd1"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": "## 2. Basic DNN with Regularization\n",
   "id": "25009ef42ca08199"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": [
    "model_reg = BasicDNN() # Same architecture\n",
    "summary(model_reg, input_size=(batch_size, 1, 28, 28))\n",
    "\n",
    "print(\"Training Basic DNN with L2 Regularization...\")\n",
    "# TF L2(0.01) corresponds to weight_decay=0.01 in PyTorch Adam\n",
    "history_reg = train_model(model_reg, train_loader, val_loader, epochs=10, weight_decay=0.01)\n",
    "all_histories.append(('Basic DNN reg.', history_reg))\n"
   ],
   "id": "dd148f77097f9f7d"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": "evaluate_model(model_reg, test_loader)\n",
   "id": "e3ea55aaf6be60b4"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": "## 3. DNN Reg. Long Training\n",
   "id": "8a1ff7b34b5b271a"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": [
    "model_long = BasicDNN() # Same architecture\n",
    "summary(model_long, input_size=(batch_size, 1, 28, 28))\n",
    "\n",
    "print(\"Training Basic DNN with L2 Regularization (25 epochs)...\")\n",
    "history_long = train_model(model_long, train_loader, val_loader, epochs=25, weight_decay=0.01)\n",
    "all_histories.append(('DNN reg. long training.', history_long))\n"
   ],
   "id": "5c2a0988eb85db23"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": "evaluate_model(model_long, test_loader)\n",
   "id": "e44581002b68f29e"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": "## 4. CNN\n",
   "id": "539bdf5bd47ba367"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": [
    "class CNN(nn.Module):\n",
    "    def __init__(self):\n",
    "        super(CNN, self).__init__()\n",
    "        # Input: (batch, 1, 28, 28)\n",
    "        self.conv1 = nn.Conv2d(1, 32, kernel_size=3) # Output: (batch, 32, 26, 26)\n",
    "        self.relu1 = nn.ReLU()\n",
    "        self.pool1 = nn.MaxPool2d(kernel_size=2)     # Output: (batch, 32, 13, 13)\n",
    "        self.conv2 = nn.Conv2d(32, 64, kernel_size=3) # Output: (batch, 64, 11, 11)\n",
    "        self.relu2 = nn.ReLU()\n",
    "        self.pool2 = nn.MaxPool2d(kernel_size=2)     # Output: (batch, 64, 5, 5)\n",
    "        self.flatten = nn.Flatten()\n",
    "        self.dropout = nn.Dropout(0.5)\n",
    "        self.fc = nn.Linear(64 * 5 * 5, 10)\n",
    "        \n",
    "    def forward(self, x):\n",
    "        x = self.relu1(self.conv1(x))\n",
    "        x = self.pool1(x)\n",
    "        x = self.relu2(self.conv2(x))\n",
    "        x = self.pool2(x)\n",
    "        x = self.flatten(x)\n",
    "        x = self.dropout(x)\n",
    "        x = self.fc(x)\n",
    "        return x\n",
    "\n",
    "model_cnn = CNN()\n",
    "summary(model_cnn, input_size=(batch_size, 1, 28, 28))\n"
   ],
   "id": "3c3ccd32330a5cf7"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": [
    "print(\"Training CNN (25 epochs)...\")\n",
    "history_cnn = train_model(model_cnn, train_loader, val_loader, epochs=25)\n",
    "all_histories.append(('CNN', history_cnn))\n"
   ],
   "id": "1c0b9f153838af84"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": "evaluate_model(model_cnn, test_loader)\n",
   "id": "6c5f25e1f076a627"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": "## History Comparison\n",
   "id": "e278e4ca072511cc"
  },
  {
   "metadata": {},
   "cell_type": "code",
   "outputs": [],
   "execution_count": null,
   "source": [
    "metrics = ['loss', 'sparse_categorical_accuracy']\n",
    "\n",
    "for metric in metrics:\n",
    "    plt.figure(figsize=(10, 6))\n",
    "    for name, history in all_histories:\n",
    "        plt.plot(history[metric], label=f'{name} (train)')\n",
    "        plt.plot(history[f'val_{metric}'], '--', label=f'{name} (val)')\n",
    "    plt.title(f'Model Comparison - {metric}')\n",
    "    plt.xlabel('Epoch')\n",
    "    plt.ylabel(metric)\n",
    "    plt.legend()\n",
    "    plt.show()\n"
   ],
   "id": "e85832ea270cdc1b"
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 2
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython2",
   "version": "2.7.6"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}